# AutoVision — Build a Custom Classifier

This notebook breaks the pipeline into individual steps so you can see
what's happening at each stage and tune things to your use case:

- **Scrape** exactly the images you want
- **Configure** training with a `TrainConfig` dataclass
- **Train** and watch per-epoch metrics
- **Predict** on new images

Swap in your own categories — anything you can search for, you can classify.

In [ ]:
import sys
sys.path.insert(0, "..")  # only needed if you haven't done `pip install -e ..`

## Step 1 — Scrape images

Uses DuckDuckGo image search — no browser, no API key, no Selenium.
Images land in `images/<category_name>/`.

Tune `n_images` to taste: 80–150 per class is usually plenty for transfer learning.

In [ ]:
from autovision.scraper import ImageScraper

scraper = ImageScraper(images_dir="images")

categories = ["sports car", "pickup truck", "minivan"]

for category in categories:
    scraper.search_and_download(category, n_images=80)

## Step 2 — Configure training

All hyperparameters live in a single `TrainConfig` object — easy to inspect,
copy, and reproduce.

Try `backbone="resnet18"` for a lighter model, or set `freeze_backbone=False`
if you have 200+ images per class and want to fine-tune the whole network.

In [ ]:
from autovision.config import TrainConfig

cfg = TrainConfig(
    images_dir="images",
    model_path="car_classifier.pt",
    backbone="efficientnet_b0",  # or "resnet18"
    freeze_backbone=True,         # only train the classifier head
    epochs=10,
    batch_size=32,
    lr=1e-3,
)

print(cfg)

## Step 3 — Train

Trains for `epochs` cycles, prints loss + accuracy per epoch.
The best checkpoint (by val accuracy) is saved automatically.

A `*` next to a row means a new best was saved that epoch.

In [ ]:
from autovision.trainer import train

model, classes = train(cfg)
print(f"\nClasses: {classes}")

## Step 4 — Predict on new images

The checkpoint is loaded fresh each call — no need to keep the model in memory.

In [ ]:
from autovision.trainer import predict

# Replace with your own image path
results = predict("path/to/car.jpg", model_path="car_classifier.pt", top_k=3)

print(f"{'Class':<22} {'Confidence':>10}")
print("-" * 35)
for cls, conf in results:
    print(f"{cls:<22} {conf:>9.1%}")

## Step 5 — Launch the Gradio demo

Opens a web UI at `http://localhost:7860` — drag any image in to see confidence scores.

In [ ]:
from app import build_demo

build_demo(model_path="car_classifier.pt").launch()